In [125]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Importando os dois datasets diretamente do site

In [25]:
import urllib.request
import zipfile
import io

# 1. URL do arquivo ZIP na UCI
url = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"

# 2. Baixar o ZIP em memória
req = urllib.request.urlopen(url)
zip_bytes = io.BytesIO(req.read())

# 3. Abrir o ZIP e inspecionar/extrair os arquivos
with zipfile.ZipFile(zip_bytes, 'r') as z:
    print("Arquivos encontrados no ZIP:", z.namelist())

    # Caso os CSVs estejam dentro de um sub-zip
    if 'student.zip' in z.namelist():
        sub_zip_bytes = io.BytesIO(z.read('student.zip'))
        with zipfile.ZipFile(sub_zip_bytes, 'r') as sub_z:
            print("Arquivos no student.zip:", sub_z.namelist())
            df_mat = pd.read_csv(sub_z.open('student-mat.csv'), sep=';')
            df_por = pd.read_csv(sub_z.open('student-por.csv'), sep=';')
    else:
        df_mat = pd.read_csv(z.open('student-mat.csv'), sep=';')
        df_por = pd.read_csv(z.open('student-por.csv'), sep=';')

print(f"\nMatemática carregado: {df_mat.shape}")
print(f"Português carregado: {df_por.shape}")

Arquivos encontrados no ZIP: ['.student.zip_old', 'student.zip']
Arquivos no student.zip: ['student-mat.csv', 'student-por.csv', 'student-merge.R', 'student.txt']

Matemática carregado: (395, 33)
Português carregado: (649, 33)


# Tratamento dos dados

In [73]:
# Criando a coluna de disciplina
df_por['disciplina'] = 'Português'
df_mat['disciplina'] = 'Matemática'

In [86]:
# Juntando as duas disciplinas
df_bruto = pd.concat([df_mat, df_por], ignore_index=True)

# Criando a cópia que sofrerá alterações
df = df_bruto.copy()

In [87]:
df

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,disciplina
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,Matemática
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,Matemática
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,Matemática
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,Matemática
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,Matemática
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1039,MS,F,19,R,GT3,T,2,3,services,other,...,4,2,1,2,5,4,10,11,10,Português
1040,MS,F,18,U,LE3,T,3,1,teacher,services,...,3,4,1,1,1,4,15,15,16,Português
1041,MS,F,18,U,GT3,T,1,1,other,other,...,1,1,1,1,5,6,11,12,9,Português
1042,MS,M,17,U,LE3,T,3,1,services,services,...,4,5,3,4,2,6,10,10,10,Português


In [117]:
print(f'Quantidade de registros duplicados: {df.duplicated().sum()}')
print(f'Quantidade de registros nulos: {df.isna().sum().sum()}')

Quantidade de registros duplicados: 0
Quantidade de registros nulos: 0


In [89]:
# Removendo colunas desnecessárias para a análise
df = df.drop(columns=[
    'sex',
    'address',
    'famsize',
    'Pstatus',
    'Medu',
    'Fedu',
    'Mjob',
    'Fjob',
    'reason',
    'guardian',
    'traveltime',
    'studytime',
    'schoolsup',
    'famsup',
    'paid',
    'activities',
    'nursery',
    'higher',
    'internet',
    'romantic',
    'famrel',
    'freetime',
    'goout',
    'Dalc',
    'Walc',
    'health'
])

In [90]:
# Renomeando colunas para a nomenclatura do sistema
df = df.rename(columns={
    'school': 'escola',
    'age': 'idade',
    'failures': 'historico_reprovacao',
    'absences': 'ausencias',
    'G1': 'nota_t1',
    'G2': 'nota_t2',
    'G3': 'nota_t3',
})

In [91]:
# Criando um identificador para cada aluno
df['id_aluno'] = [f'ALU_{i + 1:04d}' for i in range(len(df))]

In [92]:
# Reordenando as colunas
df = df[[
    'id_aluno',
    'escola',
    'disciplina',
    'idade',
    'historico_reprovacao',
    'ausencias',
    'nota_t1',
    'nota_t2',
    'nota_t3'
]]

In [93]:
df.head()

,id_aluno,escola,disciplina,idade,historico_reprovacao,ausencias,nota_t1,nota_t2,nota_t3
0,ALU_0001,GP,Matemática,18,0,6,5,6,6
1,ALU_0002,GP,Matemática,17,0,4,5,5,6
2,ALU_0003,GP,Matemática,15,3,10,7,8,10
3,ALU_0004,GP,Matemática,15,0,2,15,14,15
4,ALU_0005,GP,Matemática,16,0,4,6,10,10


In [94]:
df['nota_t1'] = (df['nota_t1'] / 2).round(1)
df['nota_t2'] = (df['nota_t2'] / 2).round(1)
df['nota_t3'] = (df['nota_t3'] / 2).round(1)

df['media_geral'] = ((df['nota_t1'] + df['nota_t2'] + df['nota_t3']) / 3).round(1)

In [98]:
df['frequencia_percentual'] = (((200 - df['ausencias']) / 200) * 100).clip(0, 100).round(1)

In [114]:
df.sort_values(by='nota_t3').head()

,id_aluno,escola,disciplina,idade,historico_reprovacao,ausencias,nota_t1,nota_t2,nota_t3,media_geral,frequencia_percentual,situacao_final
1021,ALU_1022,MS,Português,18,0,0,3.5,2.5,0.0,2.0,100.0,Evadido
914,ALU_0915,MS,Português,16,0,0,4.0,3.5,0.0,2.5,100.0,Evadido
835,ALU_0836,MS,Português,16,0,0,3.5,0.0,0.0,1.2,100.0,Evadido
131,ALU_0132,GP,Matemática,15,0,0,4.0,0.0,0.0,1.3,100.0,Evadido
135,ALU_0136,GP,Matemática,15,0,0,5.5,0.0,0.0,1.8,100.0,Evadido


## Dividindo os registros em 3 categorias: Aprovado, Reprovado e Evadido

A condição para o aluno ser reprovado é sua média geral ser menor que 6

Já a condição para considerar seu status como evadido é ele ter ido a menos do que 75% das aulas do ano. Porém, se o aluno tiver desistido da disciplina, o sistema da escola para de computar suas faltas, podendo manter sua frequência como 100% mesmo que não tenha feito as provas. Por isso, achei uma boa ideia adicionar um OU na condição para que se ele não tiver feito a prova final, ele também seja considerado como evadido.

In [123]:
condicoes = [
    (df['frequencia_percentual'] < 75.0) | (df['nota_t3'] == 0.0),
    (df['media_geral'] < 5.0)
]
escolhas = ['Evadido', 'Reprovado']

df['situacao_final'] = np.select(condicoes, escolhas, default='Aprovado')

In [124]:
df['situacao_final'].value_counts()

,count
situacao_final,
Aprovado,722
Reprovado,266
Evadido,56


# Análise Exploratória